# Notebook 04 - Visualizations

**Requirement 6:** Tables and visualizations for data exploration.

This notebook covers:
- 4-panel dashboard (boxplot, violin, bar chart, scatter)
- Summary table per city
- Interactive geographic map with Folium (**Bonus B6: Creativity**)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns, folium
from pathlib import Path
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
Path('../data').mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
df = pd.read_csv('../data/inserate_bereinigt.csv')
print(f'Loaded: {len(df)} listings from {df["stadt"].nunique()} cities')

## 1. 4-Panel Dashboard

In [ ]:
fig = plt.figure(figsize=(16,12))
gs = gridspec.GridSpec(2,2,figure=fig,hspace=0.35,wspace=0.3)
city_order = df.groupby('stadt')['preis_chf'].median().sort_values(ascending=False).index

# Panel 1: Boxplot - price distribution by city
ax1 = fig.add_subplot(gs[0,0])
sns.boxplot(data=df,y='stadt',x='preis_chf',order=city_order,palette='husl',ax=ax1)
ax1.set_title('Price Distribution by City',fontweight='bold')
ax1.set_xlabel('Rental Price (CHF/month)'); ax1.set_ylabel('')
ax1.axvline(df['preis_chf'].median(),color='red',ls='--',alpha=0.7,
            label=f"Median CHF {df['preis_chf'].median():.0f}")
ax1.legend(fontsize=9)

# Panel 2: Violin plot - price by room group
ax2 = fig.add_subplot(gs[0,1])
room_col = 'room_group' if 'room_group' in df.columns else 'zimmer_gruppe'
if room_col in df.columns:
    options = ['1-1.5 rooms','2-2.5 rooms','3-3.5 rooms','4+ rooms',
               '1-1.5 Zi','2-2.5 Zi','3-3.5 Zi','4+ Zi']
    zo = [z for z in options if z in df[room_col].values]
    sns.violinplot(data=df,x=room_col,y='preis_chf',order=zo,palette='muted',ax=ax2,inner='quartile')
ax2.set_title('Price by Room Group',fontweight='bold')
ax2.set_xlabel('Room Group'); ax2.set_ylabel('Rental Price (CHF/month)')

# Panel 3: Bar chart - average price/m2 by city
ax3 = fig.add_subplot(gs[1,0])
pm2 = df.groupby('stadt')['preis_pro_m2'].mean().sort_values(ascending=False)
bars = ax3.barh(pm2.index, pm2.values, color=sns.color_palette('husl',len(pm2)))
ax3.set_title('Average Price per m2 by City',fontweight='bold')
ax3.set_xlabel('CHF/m2')
for bar,val in zip(bars,pm2.values):
    ax3.text(val+0.3,bar.get_y()+bar.get_height()/2,f'CHF {val:.1f}',va='center',fontsize=9)

# Panel 4: Scatter - floor area vs. price
ax4 = fig.add_subplot(gs[1,1])
for s in city_order:
    sub=df[df['stadt']==s]
    ax4.scatter(sub['flaeche_m2'],sub['preis_chf'],alpha=0.5,s=25,label=s)
ax4.set_title('Floor Area vs. Rental Price',fontweight='bold')
ax4.set_xlabel('Floor Area (m2)'); ax4.set_ylabel('Rental Price (CHF/month)')
ax4.legend(fontsize=8,ncol=2)

fig.suptitle('Swiss Real Estate Market - Overview Dashboard',fontsize=15,fontweight='bold',y=1.01)
plt.savefig('../data/dashboard.png',dpi=150,bbox_inches='tight')
plt.show()
print('Saved: data/dashboard.png')

## 2. Summary Table per City

In [ ]:
table = df.groupby('stadt').agg(
    n=('preis_chf','count'), avg_price=('preis_chf','mean'),
    median=('preis_chf','median'), avg_area=('flaeche_m2','mean'),
    avg_rooms=('zimmer_anzahl','mean'), avg_m2=('preis_pro_m2','mean')
).round(1).sort_values('avg_price',ascending=False)
table.columns = ['n','Avg Price (CHF)','Median (CHF)','Avg Area (m2)','Avg Rooms','Avg CHF/m2']
print('Summary Table:')
table

## 3. Interactive Folium Map (Bonus B6: Creativity)

Geographic map of Switzerland with one circle marker per city.
- **Circle size** = number of listings
- **Color** = price level (green = affordable, orange = mid-range, red = expensive)
- **Click** on a marker to see average price and listing count

In [ ]:
coordinates = {
    'Zuerich':(47.3769,8.5417),'Genf':(46.2044,6.1432),'Bern':(46.9481,7.4474),
    'Basel':(47.5596,7.5886),'Luzern':(47.0502,8.3093),'Lausanne':(46.5197,6.6323),
    'Winterthur':(47.5003,8.7238),'St. Gallen':(47.4245,9.3767),
    'Lugano':(46.0037,8.9511),'Biel':(47.1368,7.2467),
}
city_stats = df.groupby('stadt')['preis_chf'].agg(['mean','count']).round(0)
m = folium.Map(location=[46.8182,8.2275],zoom_start=8,tiles='CartoDB positron')
prices = [city_stats.loc[c,'mean'] for c in coordinates if c in city_stats.index]
mi,ma = min(prices),max(prices)

def price_color(p):
    r=(p-mi)/(ma-mi) if ma>mi else 0.5
    return 'green' if r<0.33 else 'orange' if r<0.66 else 'red'

for city,(lat,lon) in coordinates.items():
    if city not in city_stats.index: continue
    avg=city_stats.loc[city,'mean']; n=int(city_stats.loc[city,'count'])
    folium.CircleMarker(
        location=[lat,lon], radius=12+n*0.05,
        color=price_color(avg), fill=True, fill_color=price_color(avg), fill_opacity=0.7,
        popup=folium.Popup(f'<b>{city}</b><br>Avg: CHF {avg:,.0f}/mo.<br>n={n}',max_width=180),
        tooltip=f'{city}: CHF {avg:,.0f}'
    ).add_to(m)

m.save('../data/real_estate_map.html')
print('Map saved: data/real_estate_map.html')
print('Legend: green=affordable  orange=mid-range  red=expensive')
m